In [7]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
!pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv gradio

In [8]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Any
#from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
# 환경변수 로드
#load_dotenv()
# 모델 설정
LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
llm = ChatOpenAI(model=LLM_MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [ ]:
# 검색 평가 메트릭,
# 생성 평가 메트릭

In [4]:
documents = [
    {
        "doc_id": "doc_001",
        "title": "트랜스포머 아키텍처",
        "content": "트랜스포머는 2017년 'Attention is All You Need' 논문에서 제안된 아키텍처입니다. "
                   "셀프 어텐션 메커니즘을 사용하여 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다. "
                   "인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다."
    },
    {
        "doc_id": "doc_002",
        "title": "RAG 시스템 개요",
        "content": "RAG(Retrieval-Augmented Generation)는 검색과 생성을 결합한 기법입니다. "
                   "외부 지식 베이스에서 관련 문서를 검색한 후, 이를 컨텍스트로 활용하여 LLM이 답변을 생성합니다. "
                   "환각(hallucination)을 줄이고 최신 정보를 반영할 수 있는 장점이 있습니다."
    },
    {
        "doc_id": "doc_003",
        "title": "벡터 임베딩과 유사도 검색",
        "content": "벡터 임베딩은 텍스트를 고차원 벡터 공간에 매핑하는 기술입니다. "
                   "코사인 유사도를 사용하여 의미적으로 유사한 문서를 검색합니다. "
                   "FAISS, Pinecone 등의 벡터 데이터베이스가 대규모 검색에 활용됩니다."
    },
    {
        "doc_id": "doc_004",
        "title": "프롬프트 엔지니어링",
        "content": "프롬프트 엔지니어링은 LLM에게 효과적인 지시를 설계하는 기술입니다. "
                   "Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있습니다. "
                   "시스템 프롬프트와 사용자 프롬프트를 구분하여 역할과 지시를 명확히 합니다."
    },
    {
        "doc_id": "doc_005",
        "title": "파인튜닝과 전이학습",
        "content": "파인튜닝은 사전 학습된 모델을 특정 태스크에 맞게 추가 학습시키는 기법입니다. "
                   "LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다. "
                   "전이학습을 통해 적은 데이터로도 높은 성능을 달성할 수 있습니다."
    },
    {
        "doc_id": "doc_006",
        "title": "토큰화와 텍스트 전처리",
        "content": "토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. "
                   "BPE(Byte Pair Encoding), WordPiece, SentencePiece 등의 알고리즘이 있습니다. "
                   "한국어는 교착어 특성상 형태소 분석 기반 토큰화가 효과적입니다."
    },
    {
        "doc_id": "doc_007",
        "title": "LLM 평가 메트릭",
        "content": "LLM 평가에는 자동 메트릭과 인간 평가가 사용됩니다. "
                   "BLEU, ROUGE, BERTScore 등의 자동 메트릭은 참조 답변과의 유사도를 측정합니다. "
                   "LLM-as-Judge 방식은 다른 LLM을 활용하여 품질을 평가하는 최신 접근법입니다."
    },
    {
        "doc_id": "doc_008",
        "title": "청킹 전략",
        "content": "청킹은 긴 문서를 적절한 크기로 분할하는 전략입니다. "
                   "고정 크기 청킹, 의미 기반 청킹, 재귀적 청킹 등의 방법이 있습니다. "
                   "청크 크기와 오버랩은 검색 품질에 큰 영향을 미치며, 보통 500-1000 토큰을 사용합니다."
    },
    {
        "doc_id": "doc_009",
        "title": "하이브리드 검색",
        "content": "하이브리드 검색은 키워드 검색(BM25)과 벡터 검색을 결합한 방식입니다. "
                   "RRF(Reciprocal Rank Fusion)를 통해 두 검색 결과를 효과적으로 병합합니다. "
                   "키워드 매칭의 정확성과 시맨틱 검색의 의미 이해를 동시에 활용합니다."
    },
    {
        "doc_id": "doc_010",
        "title": "멀티모달 AI",
        "content": "멀티모달 AI는 텍스트, 이미지, 오디오 등 다양한 데이터 유형을 처리합니다. "
                   "GPT-4V, Gemini 등이 대표적인 멀티모달 모델입니다. "
                   "CLIP 모델은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다."
    },
]


In [5]:
qa_dataset = [
    {
        "query_id": "q01",
        "question": "트랜스포머의 핵심 메커니즘은 무엇인가요?",
        "relevant_doc_ids": ["doc_001"],
        "ground_truth": "트랜스포머의 핵심 메커니즘은 셀프 어텐션으로, 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다."
    },
    {
        "query_id": "q02",
        "question": "RAG 시스템의 장점은 무엇인가요?",
        "relevant_doc_ids": ["doc_002"],
        "ground_truth": "RAG는 환각을 줄이고 최신 정보를 반영할 수 있으며, 외부 지식 베이스를 활용하여 더 정확한 답변을 생성합니다."
    },
    {
        "query_id": "q03",
        "question": "벡터 유사도 검색에 사용되는 데이터베이스는?",
        "relevant_doc_ids": ["doc_003"],
        "ground_truth": "FAISS, Pinecone 등의 벡터 데이터베이스가 대규모 벡터 유사도 검색에 활용됩니다."
    },
    {
        "query_id": "q04",
        "question": "프롬프트 엔지니어링의 주요 기법은?",
        "relevant_doc_ids": ["doc_004"],
        "ground_truth": "Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있으며, 시스템/사용자 프롬프트를 구분합니다."
    },
    {
        "query_id": "q05",
        "question": "효율적 파인튜닝 기법에는 어떤 것이 있나요?",
        "relevant_doc_ids": ["doc_005"],
        "ground_truth": "LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다."
    },
    {
        "query_id": "q06",
        "question": "한국어 토큰화에 적합한 방법은?",
        "relevant_doc_ids": ["doc_006"],
        "ground_truth": "한국어는 교착어 특성상 형태소 분석 기반 토큰화가 효과적입니다."
    },
    {
        "query_id": "q07",
        "question": "LLM-as-Judge란 무엇인가요?",
        "relevant_doc_ids": ["doc_007"],
        "ground_truth": "LLM-as-Judge는 다른 LLM을 활용하여 생성 품질을 평가하는 최신 접근법입니다."
    },
    {
        "query_id": "q08",
        "question": "적절한 청크 크기는 얼마인가요?",
        "relevant_doc_ids": ["doc_008"],
        "ground_truth": "보통 500-1000 토큰 크기를 사용하며, 청크 크기와 오버랩이 검색 품질에 큰 영향을 미칩니다."
    },
    {
        "query_id": "q09",
        "question": "하이브리드 검색에서 결과를 병합하는 방법은?",
        "relevant_doc_ids": ["doc_009"],
        "ground_truth": "RRF(Reciprocal Rank Fusion)를 통해 키워드 검색과 벡터 검색 결과를 효과적으로 병합합니다."
    },
    {
        "query_id": "q10",
        "question": "검색과 생성을 결합하여 환각을 줄이는 기법은?",
        "relevant_doc_ids": ["doc_002", "doc_003"],
        "ground_truth": "RAG(Retrieval-Augmented Generation)는 검색으로 관련 문서를 찾고 LLM이 이를 기반으로 답변을 생성하여 환각을 줄입니다."
    },
    {
        "query_id": "q11",
        "question": "CLIP 모델의 활용 분야는?",
        "relevant_doc_ids": ["doc_010"],
        "ground_truth": "CLIP은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다."
    },
    {
        "query_id": "q12",
        "question": "트랜스포머의 구조는 어떻게 되어있나요?",
        "relevant_doc_ids": ["doc_001"],
        "ground_truth": "인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다."
    },
]

In [9]:
for doc in documents:
  print(f"{doc['doc_id']} : {doc['title']}")

doc_001 : 트랜스포머 아키텍처
doc_002 : RAG 시스템 개요
doc_003 : 벡터 임베딩과 유사도 검색
doc_004 : 프롬프트 엔지니어링
doc_005 : 파인튜닝과 전이학습
doc_006 : 토큰화와 텍스트 전처리
doc_007 : LLM 평가 메트릭
doc_008 : 청킹 전략
doc_009 : 하이브리드 검색
doc_010 : 멀티모달 AI


In [13]:
for qa in qa_dataset[:5]:
  print(f"{qa['query_id']} : {qa['question'][:50]} ---> {qa['relevant_doc_ids']}")

q01 : 트랜스포머의 핵심 메커니즘은 무엇인가요? ---> ['doc_001']
q02 : RAG 시스템의 장점은 무엇인가요? ---> ['doc_002']
q03 : 벡터 유사도 검색에 사용되는 데이터베이스는? ---> ['doc_003']
q04 : 프롬프트 엔지니어링의 주요 기법은? ---> ['doc_004']
q05 : 효율적 파인튜닝 기법에는 어떤 것이 있나요? ---> ['doc_005']


In [14]:
embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)
langchain_docs = [
    Document(
        page_content = doc['content'],
        metadata = {'doc_id': doc['doc_id'], 'title': doc['title']}

    ) for doc in documents

]

vectorstore = FAISS.from_documents(langchain_docs, embeddings)

In [15]:
vectorstore.index.ntotal

10

In [17]:
def search_documents(query, k):
  results = vectorstore.similarity_search_with_score(query, k=k)
  retrieved = []
  for doc, score in results:
    retrieved.append({
        'doc_id' : doc.metadata['doc_id'],
                      'title':doc.metadata['title'],
        'content' : doc.page_content,
        'score' : float(score)
    })

  return retrieved

In [18]:
search_results_cache = {}
for qa in qa_dataset:
  results = search_documents(qa['question'], k=5)

  search_results_cache[qa['query_id']] = results

# 아마, qa_dataset의 qa마다 유사한 답변을 가져오는것 같음
# score는 거리이며 낮을수록 유사

In [19]:
search_results_cache

{'q01': [{'doc_id': 'doc_001',
   'title': '트랜스포머 아키텍처',
   'content': "트랜스포머는 2017년 'Attention is All You Need' 논문에서 제안된 아키텍처입니다. 셀프 어텐션 메커니즘을 사용하여 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다. 인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다.",
   'score': 0.8629103899002075},
  {'doc_id': 'doc_010',
   'title': '멀티모달 AI',
   'content': '멀티모달 AI는 텍스트, 이미지, 오디오 등 다양한 데이터 유형을 처리합니다. GPT-4V, Gemini 등이 대표적인 멀티모달 모델입니다. CLIP 모델은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다.',
   'score': 1.3324825763702393},
  {'doc_id': 'doc_005',
   'title': '파인튜닝과 전이학습',
   'content': '파인튜닝은 사전 학습된 모델을 특정 태스크에 맞게 추가 학습시키는 기법입니다. LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다. 전이학습을 통해 적은 데이터로도 높은 성능을 달성할 수 있습니다.',
   'score': 1.4281699657440186},
  {'doc_id': 'doc_009',
   'title': '하이브리드 검색',
   'content': '하이브리드 검색은 키워드 검색(BM25)과 벡터 검색을 결합한 방식입니다. RRF(Reciprocal Rank Fusion)를 통해 두 검색 결과를 효과적으로 병합합니다. 키워드 매칭의 정확성과 시맨틱 검색의 의미 이해를 동시에 활용합니다.',
   'score': 1.5083041191101074},
  {'doc_id': 'doc_004',
   'title': '프롬프트 엔지니어링',
   

In [20]:
sample_q = qa_dataset[3]
sample_q

{'query_id': 'q04',
 'question': '프롬프트 엔지니어링의 주요 기법은?',
 'relevant_doc_ids': ['doc_004'],
 'ground_truth': 'Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있으며, 시스템/사용자 프롬프트를 구분합니다.'}

In [22]:
for i, r in enumerate(search_results_cache[sample_q['query_id']]):
  print(f" {i+1}, {r['doc_id'], {r['title']}}")

 1, ('doc_004', {'프롬프트 엔지니어링'})
 2, ('doc_001', {'트랜스포머 아키텍처'})
 3, ('doc_005', {'파인튜닝과 전이학습'})
 4, ('doc_002', {'RAG 시스템 개요'})
 5, ('doc_010', {'멀티모달 AI'})


In [23]:
# 실습

new_documents = [
    {
        "doc_id": "doc_011",
        "title": "어텐션 메커니즘의 변형",
        "content": "어텐션 메커니즘에는 다양한 변형이 있습니다. "
                   "Flash Attention은 메모리 효율적인 어텐션 계산을 제공하며, "
                   "Sparse Attention은 긴 시퀀스 처리에 효과적입니다."
    },
    {
        "doc_id": "doc_012",
        "title": "LLM 양자화 기법",
        "content": "양자화는 모델의 가중치를 낮은 비트로 표현하여 메모리와 연산을 줄이는 기법입니다. "
                   "GPTQ, AWQ, GGUF 등의 포맷이 있으며, 4비트 양자화가 널리 사용됩니다."
    }
]
new_qa_pairs = [
    {
        "query_id": "q13",
        "question": "메모리 효율적인 어텐션 방법은?",
        "relevant_doc_ids": ["doc_011", "doc_001"],
        "ground_truth": "Flash Attention은 메모리 효율적인 어텐션 계산을 제공합니다."
    },
    {
        "query_id": "q14",
        "question": "LLM 모델 크기를 줄이는 양자화 포맷은?",
        "relevant_doc_ids": ["doc_012"],
        "ground_truth": "GPTQ, AWQ, GGUF 등의 양자화 포맷이 있으며 4비트 양자화가 널리 사용됩니다."
    }
]

In [25]:
# 기존 qa_pair, document에 더하고 search_results_cache 업데이트

In [27]:
qa_dataset.extend(new_qa_pairs)
documents.extend(new_documents)

In [36]:
search_results_cache.get('q14')

[{'doc_id': 'doc_007',
  'title': 'LLM 평가 메트릭',
  'content': 'LLM 평가에는 자동 메트릭과 인간 평가가 사용됩니다. BLEU, ROUGE, BERTScore 등의 자동 메트릭은 참조 답변과의 유사도를 측정합니다. LLM-as-Judge 방식은 다른 LLM을 활용하여 품질을 평가하는 최신 접근법입니다.',
  'score': 1.0641818046569824},
 {'doc_id': 'doc_004',
  'title': '프롬프트 엔지니어링',
  'content': '프롬프트 엔지니어링은 LLM에게 효과적인 지시를 설계하는 기술입니다. Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있습니다. 시스템 프롬프트와 사용자 프롬프트를 구분하여 역할과 지시를 명확히 합니다.',
  'score': 1.369309663772583},
 {'doc_id': 'doc_005',
  'title': '파인튜닝과 전이학습',
  'content': '파인튜닝은 사전 학습된 모델을 특정 태스크에 맞게 추가 학습시키는 기법입니다. LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다. 전이학습을 통해 적은 데이터로도 높은 성능을 달성할 수 있습니다.',
  'score': 1.391709327697754},
 {'doc_id': 'doc_008',
  'title': '청킹 전략',
  'content': '청킹은 긴 문서를 적절한 크기로 분할하는 전략입니다. 고정 크기 청킹, 의미 기반 청킹, 재귀적 청킹 등의 방법이 있습니다. 청크 크기와 오버랩은 검색 품질에 큰 영향을 미치며, 보통 500-1000 토큰을 사용합니다.',
  'score': 1.487743854522705},
 {'doc_id': 'doc_010',
  'title': '멀티모달 AI',
  'content': '멀티모달 AI는 텍스트, 이미지, 오디오 등 다양한 데이터 유형을 처리합니다. GP

In [30]:
search_results_cache = {}
for qa in qa_dataset:
  results = search_documents(qa['question'], k=5)

  search_results_cache[qa['query_id']] = results

In [ ]:
search_results_cache

In [ ]:
# qa_dataset
# documents



In [37]:
# 내가 작성한 함수, documents 를 넣고, 신규 qa_pair를 넣었을 때
# q_id와 관련된 doc(rel_docs)이 존재하면 True, 아니면 False 반환

def validate_dataset(docs, qa_pairs):
    results = {}

    for qa in qa_pairs:
        q_id = qa["query_id"]
        is_valid = True

        for rel_id in qa["relevant_doc_ids"]:
            found = False
            for doc in docs:
                if doc["doc_id"] == rel_id:
                    found = True
                    break
            if not found:
                is_valid = False
                break

        results[q_id] = is_valid
        print(f"{q_id}: {is_valid}")

    return results


In [41]:
validate_dataset(documents, new_qa_pairs)

q13: True
q14: True


{'q13': True, 'q14': True}

In [42]:
# all_docs = documents + new_documents
# all_qa  = qa_dataset + new_qa_pairs


In [47]:
# 정답 (강사님)

def validate_dataset(docs, qa_pairs):
  # qa_pairs[q_id] 문서가 docs에 있다면 True, 없다면 False
  doc_ids = {d['doc_id'] for d in docs}
  errors = []
  for qa in qa_pairs:
    for rid in qa['relevant_doc_ids']:
      if rid not in doc_ids:
        errors.append(f"{qa['query_id']}:{rid} 문서없음")

  if errors:
    print(f"오류")

  else:
    print(f"통과")

  return len(errors)


In [48]:
validate_dataset(documents, qa_dataset)

통과


0

In [50]:
# 검색결과에 relevant id가 나오면 pass, 안나오면 fail
# 이런식의 지표를 hit-rate라고함

# 몇개 뽑는것중에 몇개 관련된게 있는지 다 다름
# (1, 2, 3, 4, 5) 중에 1개
# 1개 중 1개
# (1, 2, ... 100)개 중에 1개..
# 이를 hitrate@k

In [74]:
def hit_rate_at_k(query_id, k):
  qa_list = [q for q in qa_dataset if q['query_id'] == query_id]
  qa = qa_list[0] if qa_list else None

  if qa is None:  # ← 이 처리가 빠져있던 것
    return False


  relevant_ids = set(qa['relevant_doc_ids'])
  retrieved = search_results_cache[query_id][:k]
  retrieved_ids = {r['doc_id'] for r in retrieved}

  return 1 if relevant_ids & retrieved_ids else 0 # relevant_ids에 있는게 retrieved_ids에 하나라도 있으면


In [58]:
def average_hit_rate_at_k(k):
  hits = [hit_rate_at_k(qa['query_id'], k) for qa in qa_dataset]


  return sum(hits)/ len(hits)

In [62]:
average_hit_rate_at_k(5)

0.9285714285714286

In [66]:
for qa in qa_dataset:
  hit = hit_rate_at_k(qa['query_id'], k=3)
  retrieved_ids = [r['doc_id'] for r in search_results_cache[qa['query_id']][:3]]
  status = 'HIT' if hit else 'MISS'
  print(f"[{status}] {qa['query_id']}: 정답 = {qa['relevant_doc_ids']} | 검색 = {retrieved_ids}")

[HIT] q01: 정답 = ['doc_001'] | 검색 = ['doc_001', 'doc_010', 'doc_005']
[HIT] q02: 정답 = ['doc_002'] | 검색 = ['doc_002', 'doc_004', 'doc_009']
[HIT] q03: 정답 = ['doc_003'] | 검색 = ['doc_003', 'doc_009', 'doc_006']
[HIT] q04: 정답 = ['doc_004'] | 검색 = ['doc_004', 'doc_001', 'doc_005']
[HIT] q05: 정답 = ['doc_005'] | 검색 = ['doc_005', 'doc_003', 'doc_004']
[HIT] q06: 정답 = ['doc_006'] | 검색 = ['doc_006', 'doc_009', 'doc_008']
[HIT] q07: 정답 = ['doc_007'] | 검색 = ['doc_007', 'doc_004', 'doc_002']
[HIT] q08: 정답 = ['doc_008'] | 검색 = ['doc_008', 'doc_003', 'doc_001']
[HIT] q09: 정답 = ['doc_009'] | 검색 = ['doc_009', 'doc_002', 'doc_008']
[HIT] q10: 정답 = ['doc_002', 'doc_003'] | 검색 = ['doc_002', 'doc_009', 'doc_008']
[HIT] q11: 정답 = ['doc_010'] | 검색 = ['doc_010', 'doc_005', 'doc_006']
[HIT] q12: 정답 = ['doc_001'] | 검색 = ['doc_001', 'doc_010', 'doc_004']
[HIT] q13: 정답 = ['doc_011', 'doc_001'] | 검색 = ['doc_001', 'doc_005', 'doc_004']
[MISS] q14: 정답 = ['doc_012'] | 검색 = ['doc_007', 'doc_004', 'doc_005']


In [67]:
new_qa = [
    {"query_id": "q13", "question": "Flash Attention의 장점은?",
     "relevant_doc_ids": ["doc_001"],
     "ground_truth": "Flash Attention은 메모리 효율적인 어텐션 계산을 제공합니다."},
    {"query_id": "q14", "question": "양자화로 모델 크기를 줄이는 방법은?",
     "relevant_doc_ids": ["doc_005"],
     "ground_truth": "LoRA, QLoRA 등의 효율적 파인튜닝과 양자화 기법으로 모델 크기를 줄입니다."},
    {"query_id": "q15", "question": "RAG에서 청크 크기가 검색에 미치는 영향은?",
     "relevant_doc_ids": ["doc_008", "doc_002"],
     "ground_truth": "청크 크기는 검색 품질에 큰 영향을 미치며 보통 500-1000 토큰을 사용합니다."},
]

In [68]:
# 지금 올린 new_qa로??? average hit rate 계산?

def average_hit_rate_at_k(k):
  hits = [hit_rate_at_k(qa['query_id'], k) for qa in qa_dataset]


  return sum(hits)/ len(hits)

In [77]:
for qa in new_qa:
  hit = hit_rate_at_k(qa['query_id'], k=3)
  retrieved_ids = [r['doc_id'] for r in search_results_cache[qa['query_id']][:3]]
  status = 'HIT' if hit else 'MISS'

  print(f"[{status}] {qa['query_id']}: 정답 = {qa['relevant_doc_ids']} | 검색 = {retrieved_ids}")

[HIT] q13: 정답 = ['doc_001'] | 검색 = ['doc_001', 'doc_005', 'doc_004']
[MISS] q14: 정답 = ['doc_005'] | 검색 = ['doc_007', 'doc_004', 'doc_005']


KeyError: 'q15'

In [78]:
for qa in new_qa:
    hit = hit_rate_at_k(qa['query_id'], k=3)
    retrieved = search_results_cache.get(qa['query_id'], [])  # 없으면 빈 리스트
    retrieved_ids = [r['doc_id'] for r in retrieved[:3]]
    status = 'HIT' if hit else 'MISS'

    print(f"[{status}] {qa['query_id']}: 정답 = {qa['relevant_doc_ids']} | 검색 = {retrieved_ids}")

[HIT] q13: 정답 = ['doc_001'] | 검색 = ['doc_001', 'doc_005', 'doc_004']
[MISS] q14: 정답 = ['doc_005'] | 검색 = ['doc_007', 'doc_004', 'doc_005']
[MISS] q15: 정답 = ['doc_008', 'doc_002'] | 검색 = []


In [79]:
# 강사님 코드

for qa in new_qa:
  search_results_cache[qa['query_id']] = search_documents(qa['question'], k=5)

In [84]:
for qa in new_qa:
  for k in [1,3,5]:
    hit = hit_rate_at_k(qa['query_id'], k)
    status = 'HIT' if hit else 'MISS'
    print(f"{qa['query_id']} (@k = {k}) : {status} 정답: {qa['relevant_doc_ids']}")
    print(f"HR@{k}: {average_hit_rate_at_k(k)}")

q13 (@k = 1) : MISS 정답: ['doc_001']
HR@1: 0.8571428571428571
q13 (@k = 3) : HIT 정답: ['doc_001']
HR@3: 0.9285714285714286
q13 (@k = 5) : HIT 정답: ['doc_001']
HR@5: 0.9285714285714286
q14 (@k = 1) : MISS 정답: ['doc_005']
HR@1: 0.8571428571428571
q14 (@k = 3) : MISS 정답: ['doc_005']
HR@3: 0.9285714285714286
q14 (@k = 5) : MISS 정답: ['doc_005']
HR@5: 0.9285714285714286
q15 (@k = 1) : MISS 정답: ['doc_008', 'doc_002']
HR@1: 0.8571428571428571
q15 (@k = 3) : MISS 정답: ['doc_008', 'doc_002']
HR@3: 0.9285714285714286
q15 (@k = 5) : MISS 정답: ['doc_008', 'doc_002']
HR@5: 0.9285714285714286


In [ ]:
# iterator (next)

# 첫번째 코드는 찾은 결과로 리스트를 만들어버림
qa_list = [q for q in qa_dataset if q['query_id'] == query_id]

# 두번째 코드는 찾은 결과를 반환만 함
qa = next(q for q in qa_dataset if q['query_id'] == query_id)

## 2. MRR
- 평균 역수 랭킹 (Mean reciprocal (1/n) Ranking)

In [85]:
# 1/1 : 1
# 2: 1/2 : 0.5...
# 3번만에 나왔다.. 1/3 : 0.33

In [90]:
def reciprocal_rank(query_id, k):
  qa = next(q for q in qa_dataset if q['query_id'] == query_id)
  relevant_ids = set(qa['relevant_doc_ids'])
  retrieved = search_results_cache[query_id][:k]

  for rank, result in enumerate(retrieved, 1):
    # 1, result1
    # 2, result2
    # ...
    if result['doc_id'] in relevant_ids:
      return 1.0 /rank

  # 없다면 1/무한 이므로.. 그냥 0 반환
  return 0.0

def mean_reciprocal_rank(k):
  rr_score = [reciprocal_rank(qa['query_id'], k) for qa in qa_dataset]
  return sum(rr_score) / len(rr_score)

In [93]:
mean_reciprocal_rank(k=5)

0.880952380952381

In [94]:
rows = []
for qa in qa_dataset:
  rr = reciprocal_rank(qa['query_id'], 5)
  rows.append({'query_id': qa['query_id'], 'question': qa['question'], 'RR@5' : {round(rr,4)}})

df_rr = pd.DataFrame(rows)
df_rr

,query_id,question,RR@5
0,q01,트랜스포머의 핵심 메커니즘은 무엇인가요?,{1.0}
1,q02,RAG 시스템의 장점은 무엇인가요?,{1.0}
2,q03,벡터 유사도 검색에 사용되는 데이터베이스는?,{1.0}
3,q04,프롬프트 엔지니어링의 주요 기법은?,{1.0}
4,q05,효율적 파인튜닝 기법에는 어떤 것이 있나요?,{1.0}
5,q06,한국어 토큰화에 적합한 방법은?,{1.0}
6,q07,LLM-as-Judge란 무엇인가요?,{1.0}
7,q08,적절한 청크 크기는 얼마인가요?,{1.0}
8,q09,하이브리드 검색에서 결과를 병합하는 방법은?,{1.0}
9,q10,검색과 생성을 결합하여 환각을 줄이는 기법은?,{1.0}


## DCG, nDCG(normalized)

In [96]:
# DCG (rel score) / log(i+1)
# nDCG : DCG / IDCG (ideal DCG)

In [119]:
def dcg_at_k(relevances, k):
  relevances = relevances[:k]
  dcg = 0.0
  for i, rel in enumerate(relevances):
    dcg += rel / np.log2(i+2)
  return dcg

In [120]:
def ndcg_at_k(query_id, k):
  qa = next(q for q in qa_dataset if q['query_id'] == query_id)
  relevant_ids = set(qa['relevant_doc_ids'])
  retrieved = search_results_cache[query_id][:k]

  relevances = [1 if r['doc_id'] in relevant_ids else 0 for r in retrieved]
  dcg = dcg_at_k(relevances, k)

  ideal_relevances = sorted(relevances, reverse=True)
  total_relevant = len(relevant_ids)
  ideal_relevances = [1] * min(total_relevant, k) + [0] * max(0, k-total_relevant)
  idcg = dcg_at_k(ideal_relevances, k)

  if idcg == 0:
    return 0.0

  return dcg/idcg

In [107]:
sorted([1, 0, 1, 0 , 1, 0], reverse=True) # 관련도가 높은것부터 나오게하려면 resverse sorting

[1, 1, 1, 0, 0, 0]

In [117]:
def average_ndcg_at_k(k):
  scores = [ndcg_at_k(qa['query_id'], k) for qa in qa_dataset]
  return sum(scores) / len(scores)

In [122]:
for k in [1,3,5]:
  score = average_ndcg_at_k(k)
  print(f"{k} : {score}")

1 : 0.8571428571428571
3 : 0.851408627796299
5 : 0.8702706365514842


In [123]:
# Precision Recall